In [19]:
!pip install cdsapi netCDF4 cfgrib xarray

In [20]:
import cdsapi
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
import time
import shutil
import os
import time
import calendar
import gzip
import cdsapi
from datetime import datetime

In [21]:
Path("processed_data/daily_era5_chunks").mkdir(parents=True, exist_ok=True)
Path("final_datasets").mkdir(exist_ok=True)
print("Folders created!")

Folders created!


In [22]:
# Check disk space
def check_space():
    total, used, free = shutil.disk_usage(".")
    free_gb = free / (1024**3)
    print(f"Total: {total/(1024**3):.1f} GB")
    print(f"Used: {used/(1024**3):.1f} GB")
    print(f"Free: {free_gb:.1f} GB")
    return free_gb

free_space = check_space()
if free_space < 5:
    print("WARNING: Less than 5GB free!")
else:
    print("Enough space to proceed!")


Total: 19.5 GB
Used: 16.4 GB
Free: 3.1 GB


In [ ]:

Path(f"config").joinpath('.cdsapirc').write_text(
    'url: https://cds.climate.copernicus.eu/api\n'
    'key: \n'
)

print("Created .cdsapirc")

Created .cdsapirc


In [ ]:

# 1. Get API key
cds_key = input("Paste your CDS API key: ")

# 2. Create config file
config_file = Path(f"config/.cdsapirc")
config_content = f"""url: https://cds.climate.copernicus.eu/api
key: {cds_key}
"""

# 3. Write file
config_file.write_text(config_content)
os.chmod(config_file, 0o600)

# 4. Test
try:
    cds_client = cdsapi.Client()
    print("Success! CDS API ready!")
except Exception as e:
    print(f"Error: {e}")

In [ ]:

try:
    cds_client = cdsapi.Client()
    print("CDS client initialized!")
except Exception as e:
    print(f"ERROR: CDS API not configured!")
    print(f"   {e}")
    print("\n Setup instructions:")
    print("   1. Register at https://cds.climate.copernicus.eu")
    print("   2. Create ~/.cdsapirc file with credentials")
    raise

In [25]:
# BBOX = {
#     'minimum_longitude': 10,
#     'maximum_longitude': 37,
#     'minimum_latitude': 30,
#     'maximum_latitude': 39
# }

# Levantine Basin
BBOX = {
    'minimum_longitude': 24,  #westernmost point 
    'maximum_longitude': 36, #easternmost point
    'minimum_latitude': 31, #southernmost point 
    'maximum_latitude': 37 #northernmost point
}
# BBOX = {
#     'minimum_longitude': 29,  
#     'maximum_longitude': 36,
#     'minimum_latitude': 30,
#     'maximum_latitude': 36    
# }

INSTANT_VARS = [
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    '2m_temperature',
    'mean_sea_level_pressure',
    'sea_surface_temperature'
]

ACCUM_VARS = [
    'mean_surface_net_short_wave_radiation_flux',   # Solar (W/m²)
    'mean_surface_net_long_wave_radiation_flux',     # Thermal (W/m²)
    'mean_surface_sensible_heat_flux',
    'mean_surface_latent_heat_flux'
]

# ACCUM_VARS = ['surface_net_thermal_radiation','surface_net_solar_radiation']

# INSTANT_VARS = [
#     '10m_u_component_of_wind',
#     '10m_v_component_of_wind',
#     '2m_temperature',
#     'mean_sea_level_pressure',
#     'sea_surface_temperature',
#     'mean_surface_net_short_wave_radiation_flux',   # Solar (W/m²)
#     'mean_surface_net_long_wave_radiation_flux',     # Thermal (W/m²)
#     'mean_surface_sensible_heat_flux',
#     'mean_surface_latent_heat_flux'
# ]

AREA_CDS = [
    BBOX['maximum_latitude'],   # North
    BBOX['minimum_longitude'],  # West
    BBOX['minimum_latitude'],   # South
    BBOX['maximum_longitude']   # East
]

In [26]:

def detect_file_type(path: Path) -> str:
    with open(path, "rb") as f:
        head = f.read(8)
    print(head)
    # GRIB detection
    if head.startswith(b'GRIB'):
        return 'grib'
    
    # NetCDF / HDF5 detection
    if head.startswith(b'\x89HDF') or head.startswith(b'CDF\x01') or head.startswith(b'CDF\x02'):
        return 'netcdf'
    
    # Text or error messages
    if head.lstrip().startswith(b'<') or head.lstrip().startswith(b'{'):
        return 'error_text'
    
    return 'unknown'


In [30]:

with open("temp_accum_1993_01.nc", "rb") as f:
    head = f.read(8)
print(head)  # b'\x1f\x8b' → gzip


b'PK\x03\x04\x14\x00\x00\x00'


In [31]:
from zipfile import ZipFile

with ZipFile("temp_accum_1993_01.nc", 'r') as zip_ref:
    zip_ref.extractall("config")


In [27]:
def load_var_from_file(path: Path, short_name: str):
    ftype = detect_file_type(path)

    # mapping for cfgrib shortName
    short_map = {
        '10m_u_component_of_wind': 'u10',
        '10m_v_component_of_wind': 'v10',
        '2m_temperature': 't2m',
        'mean_sea_level_pressure': 'msl',
        'sea_surface_temperature': 'sst',  
        # Mean flux variables (use actual GRIB codes!)
        'mean_surface_net_short_wave_radiation_flux': 'avg_snswrf',  
        'mean_surface_net_long_wave_radiation_flux': 'avg_snlwrf',   
        'mean_surface_sensible_heat_flux': 'avg_ishf',              
        'mean_surface_latent_heat_flux': 'avg_slhtf'               
    }
        # 'mean_surface_net_short_wave_radiation_flux': 'ssr_flux',
        # 'mean_surface_net_long_wave_radiation_flux': 'str_flux',
        # 'mean_surface_sensible_heat_flux': 'sensible_flux',
        # 'mean_surface_latent_heat_flux': 'latent_flux'
        # 'surface_net_thermal_radiation': 'str', 
        # 'surface_net_solar_radiation': 'ssr'
    
    cfname = short_map.get(short_name, short_name)

    if ftype == 'netcdf':
        ds = xr.open_dataset(path)
        if short_name in ds:
            return ds[[short_name]]
        if cfname in ds:
            return ds[[cfname]]
        raise KeyError(f"{short_name} not found in NetCDF file.")

    # GRIB – filter by shortName
    ds = xr.open_dataset(
        path,
        engine="cfgrib",
        backend_kwargs={"filter_by_keys": {"shortName": cfname}}
    )

    data_vars = list(ds.data_vars)
    if not data_vars:
        raise KeyError(f"No GRIB data found for shortName={cfname}.")

    return ds[[data_vars[0]]]

In [28]:
def load_file_all_vars(path: Path, var_list: list):
    """Load all variables from a file (NetCDF or GRIB)"""
    ftype = detect_file_type(path)
    
    if ftype == 'netcdf':
        # Easy - just open it
        return xr.open_dataset(path)
    
    if ftype == 'grib':
        # Need to load each variable separately and merge
        parts = []
        for var in var_list:
            try:
                parts.append(load_var_from_file(path, var))
            except Exception as e:
                print(f"Could not load {var}: {e}")
        if not parts:
            raise ValueError(f"No variables loaded from GRIB file")
        return xr.merge(parts)
    
    raise ValueError(f"Cannot load file type: {ftype}")

In [29]:
def remove_left_tempfiles(filepath:Path):
    temp = Path(filepath)
    if temp.exists() :
        print(f"Warning: leftover temp file found → removing {temp}")
        temp.unlink()


In [30]:
def download_from_era5(Varibles:list[str],year:int, month:int,path:Path):
    last_day = calendar.monthrange(year, month)[1]

    cds_client.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "format": "netcdf",
            "variable": Varibles,
            "year": str(year),
            "month": f"{month:02d}",
            "day": [f"{d:02d}" for d in range(1, last_day + 1)],
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "area": AREA_CDS
        }
    ).download(str(path))

In [31]:
def download_temp_files(year: int, month: int,out_dir: Path):
    """
    Download the datsets as temp files before preprocessing
    """
    temp_instant = Path(f"temp_instant_{year}_{month:02d}.nc")
    temp_accum  = Path(f"temp_accum_{year}_{month:02d}.nc")   
        
    for k in [temp_instant,temp_accum]:  
        remove_left_tempfiles(k);

    # ----------------------------------------
    # DOWNLOAD
    # ----------------------------------------
    print("Downloading instant vars...")
    download_from_era5(INSTANT_VARS,year,month,temp_instant)

    print("Downloading Accum Vars...")
    download_from_era5(ACCUM_VARS,year,month,temp_accum)

    return temp_instant,temp_accum


In [32]:
def convert_temps_to_celsius(ds):
    """
    Convert temperature variables from Kelvin to Celsius
        """
    temp_vars = ['t2m', 'sst']  # Variables that need conversion
    
    for var in temp_vars:
        if var in ds:
            ds[var] = ds[var] - 273.15
            ds[var].attrs['units'] = '°C'
            print(f"{var}: K -> °C")
    
    return ds

In [33]:
def process_one_month(year: int, month: int, out_dir: Path = Path("processed_data/daily_era5_chunks")):
    
    out_dir.mkdir(parents=True, exist_ok=True)
    daily_output = out_dir / f"daily_era5_{year}_{month:02d}.nc"
    if daily_output.exists():
        print("Already processed:", daily_output)
        return daily_output

    # ----------------------------------------
    # DOWNLOAD
    # ----------------------------------------
    
    temp_instant,temp_accum = download_temp_files(year,month,out_dir)
    
    # ========================================
    # LOAD & PROCESS
    # ========================================
    print("Loading and processing data...")

    # Detect and report file types
    ftype_instant = detect_file_type(temp_instant)
    ftype_accum = detect_file_type(temp_accum)
        
    print(f"Instant Vars: {ftype_instant} ({temp_instant.stat().st_size / 1024**2:.1f} MB)")
    print(f"Accum vars: {ftype_accum} ({temp_accum.stat().st_size / 1024**2:.1f} MB)")
        

    ds_instant = load_file_all_vars(temp_instant, INSTANT_VARS)
    ds_accum = load_file_all_vars(temp_accum, ACCUM_VARS)
 
        
    # Standardize variable names
    print("Standardizing variable names...")
    rename_dict = {}
    for k in ds_instant.data_vars:
        k_lower = k.lower()
        if 'u10' in k_lower or 'u_component_of_wind' in k_lower:
            rename_dict[k] = 'u10'
        elif 'v10' in k_lower or 'v_component_of_wind' in k_lower:
            rename_dict[k] = 'v10'
        elif 't2m' in k_lower or '2m_temperature' in k_lower:
            rename_dict[k] = 't2m'
        elif 'msl' in k_lower or 'mean_sea_level' in k_lower:
            rename_dict[k] = 'msl'
        elif 'sst' in k_lower or 'sea_surface_temperature' in k_lower:
            rename_dict[k] = 'sst'
        
    ds_instant = ds_instant.rename(rename_dict)
    print(f"Instantaneous variables: {list(ds_instant.data_vars)}")

    rename_accum = {}
    for k in ds_accum.data_vars:
        k_lower = k.lower()
        
        # Solar/shortwave (avg_snswrf → ssr_flux)
        if 'snswrf' in k_lower or 'short_wave' in k_lower or 'solar' in k_lower:
            rename_accum[k] = 'ssr_flux'
        
        # Thermal/longwave (avg_snlwrf → str_flux)
        elif 'snlwrf' in k_lower or 'long_wave' in k_lower or 'thermal' in k_lower:
            rename_accum[k] = 'str_flux'
        
        # Sensible (avg_ishf → sensible_flux)
        elif 'ishf' in k_lower or 'sensible' in k_lower:
            rename_accum[k] = 'sensible_flux'
        
        # Latent (avg_slhtf → latent_flux)
        elif 'slhtf' in k_lower or 'latent' in k_lower:
            rename_accum[k] = 'latent_flux'
    
    ds_accum = ds_accum.rename(rename_accum)
    print(f" Flux vars: {list(ds_accum.data_vars)}")

    # ========================================
    # REMOVE EXTRA COORDINATES
    # ========================================
    ds_instant = ds_instant.drop_vars(['number','expver'], errors='ignore')
    ds_accum   = ds_accum.drop_vars(['number','expver'], errors='ignore')

    # ========================================
    # RENAME TIME VARIABLE
    # ========================================
    ds_instant = ds_instant.rename({"valid_time": "time"}) if "valid_time" in ds_instant.coords else ds_instant
    ds_accum   = ds_accum.rename({"valid_time": "time"}) if "valid_time" in ds_accum.coords else ds_accum

        
    print("Resampling to daily means...")
    ds_instant_daily = ds_instant.resample(time='1D').mean()
    ds_accum_daily   = ds_accum.resample(time='1D').mean()
    
    common_days = np.array(np.intersect1d(ds_instant_daily.time.values, ds_accum_daily.time.values), dtype="datetime64[ns]")
    
    ds_instant_daily = ds_instant_daily.sel(time=common_days)
    ds_accum_daily   = ds_accum_daily.sel(time=common_days)

    print("Merging to daily instant and accum data...")
    ds = xr.merge([ds_instant_daily, ds_accum_daily])



    # Derived wind speed
    print("Calculating derived variables...")
    ds["wind_speed"] = np.sqrt(ds["u10"]**2 + ds["v10"]**2)

    # Net heat flux (if you have all components)
    flux_vars = []
    if 'ssr_flux' in ds:
        flux_vars.append('ssr_flux')
    if 'str_flux' in ds:
        flux_vars.append('str_flux')
    if 'sensible_flux' in ds:
        flux_vars.append('sensible_flux')
    if 'latent_flux' in ds:
        flux_vars.append('latent_flux')
    
    if len(flux_vars) >= 2:
        ds['net_heat_flux'] = sum(ds[var] for var in flux_vars)
        print(f"net_heat_flux = {' + '.join(flux_vars)}")

    print("Converting temperatures to Celsius...")
    ds = convert_temps_to_celsius(ds)

    print("Selecting Variables")
    # Start with variables that ALWAYS exist
    output_vars = ["u10", "v10", "t2m", "msl", "sst", "wind_speed"]
    
    # Add optional variables only if they exist
    for var in ['ssr_flux', 'str_flux', 'sensible_flux', 'latent_flux', 'net_heat_flux']:
        if var in ds:
            output_vars.append(var)  #Only append if exists
    
    # Select only variables that are actually in the dataset
    ds_feat = ds[output_vars]

    print("Saving:", daily_output)
    ds_feat.to_netcdf(daily_output)

    temp_instant.unlink(missing_ok=True)
    temp_accum.unlink(missing_ok=True)

    print("Done.")
    return daily_output


In [34]:
def process_all_months(start_year=1993, end_year=2025):
    """
    Process all months from start_year to end_year
    """
    
    print(f"\nStarting: {start_year} to {end_year}")
    print(f"Region: Levantine Basin (29-36°E, 30-36°N)")
    print(f"Estimated time: 10-15 hours\n")
    
    total_months = (end_year - start_year + 1) * 12
    processed = 0
    failed = []
    
    start_time = time.time()
    
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            
            # Show progress
            if processed > 0:
                elapsed_hours = (time.time() - start_time) / 3600
                eta_hours = (elapsed_hours / processed) * (total_months - processed)
                print(f"\n[{processed}/{total_months}] {processed/total_months*100:.1f}% | ETA: {eta_hours:.1f}h")
            
            # Process
            result = process_one_month(year, month)
            
            if result:
                processed += 1
            else:
                failed.append(f"{year}-{month:02d}")
            
            # Pause 2 seconds (be nice to server)
            time.sleep(5)
    
    # Done!
    total_hours = (time.time() - start_time) / 3600
    print(f"\n{'='*65}")
    print(f" DONE!")
    print(f" Processed: {processed}/{total_months}")
    print(f" Time: {total_hours:.1f} hours")
    if failed:
        print(f"   Failed: {failed}")
    print(f"{'='*65}")
    
    return processed, failed


In [16]:
process_all_months(1993, 2025)


Starting: 1993 to 2025
Region: Levantine Basin (29-36°E, 30-36°N)
Estimated time: 10-15 hours

Already processed: processed_data/daily_era5_chunks/daily_era5_1993_01.nc

[1/396] 0.3% | ETA: 0.5h
Already processed: processed_data/daily_era5_chunks/daily_era5_1993_02.nc

[2/396] 0.5% | ETA: 0.5h
Already processed: processed_data/daily_era5_chunks/daily_era5_1993_03.nc

[3/396] 0.8% | ETA: 0.5h
Already processed: processed_data/daily_era5_chunks/daily_era5_1993_04.nc

[4/396] 1.0% | ETA: 0.5h
Already processed: processed_data/daily_era5_chunks/daily_era5_1993_05.nc

[5/396] 1.3% | ETA: 0.5h
Already processed: processed_data/daily_era5_chunks/daily_era5_1993_06.nc

[6/396] 1.5% | ETA: 0.5h
Already processed: processed_data/daily_era5_chunks/daily_era5_1993_07.nc

[7/396] 1.8% | ETA: 0.5h
Already processed: processed_data/daily_era5_chunks/daily_era5_1993_08.nc

[8/396] 2.0% | ETA: 0.5h
Already processed: processed_data/daily_era5_chunks/daily_era5_1993_09.nc

[9/396] 2.3% | ETA: 0.5h
Alre

2026-01-18 03:29:07,028 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-01-18 03:29:07,029 INFO Request ID is d7f13739-f271-4892-911e-86ac692c4800
2026-01-18 03:29:07,095 INFO status has been updated to accepted
2026-01-18 03:29:28,269 INFO status has been updated to successful


6236032e0d2c84089e0d2e8bfa14da19.nc:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

2026-01-18 03:29:29,345 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-01-18 03:29:29,346 INFO Request ID is c9c2c4d7-6b85-446e-9d9a-73d0c67552e3
2026-01-18 03:29:29,419 INFO status has been updated to accepted
2026-01-18 03:29:42,869 INFO status has been updated to running
2026-01-18 03:29:50,526 INFO status has been updated to successful


65bfa80c9a3e74ee65d68a76897361c1.nc:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

Loading and processing data...
b'\x89HDF\r\n\x1a\n'
b'\x89HDF\r\n\x1a\n'
Instant Vars: netcdf (1.3 MB)
Accum vars: netcdf (1.1 MB)
b'\x89HDF\r\n\x1a\n'
b'\x89HDF\r\n\x1a\n'
Standardizing variable names...
Instantaneous variables: ['u10', 'v10', 't2m', 'msl', 'sst']
 Flux vars: ['ssr_flux', 'str_flux', 'sensible_flux', 'latent_flux']
Resampling to daily means...
Merging to daily instant and accum data...
Calculating derived variables...
net_heat_flux = ssr_flux + str_flux + sensible_flux + latent_flux
Converting temperatures to Celsius...
t2m: K -> °C
sst: K -> °C
Selecting Variables
Saving: processed_data/daily_era5_chunks/daily_era5_2023_11.nc
Done.

[371/396] 93.7% | ETA: 0.0h


... (122 outputs left out)

2026-01-18 04:09:05,855 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-01-18 04:09:05,856 INFO Request ID is 723a27bd-84d8-485a-9a9b-5b0d6df7c320
2026-01-18 04:09:05,924 INFO status has been updated to accepted
2026-01-18 04:09:14,737 INFO status has been updated to running
2026-01-18 04:09:56,268 INFO status has been updated to successful


2686aefd2bb76290d1cbc8bf4e678b93.nc:   0%|          | 0.00/1.15M [00:00<?, ?B/s]

Loading and processing data...
b'\x89HDF\r\n\x1a\n'
b'\x89HDF\r\n\x1a\n'
Instant Vars: netcdf (1.3 MB)
Accum vars: netcdf (1.1 MB)
b'\x89HDF\r\n\x1a\n'
b'\x89HDF\r\n\x1a\n'
Standardizing variable names...
Instantaneous variables: ['u10', 'v10', 't2m', 'msl', 'sst']
 Flux vars: ['ssr_flux', 'str_flux', 'sensible_flux', 'latent_flux']
Resampling to daily means...
Merging to daily instant and accum data...
Calculating derived variables...
net_heat_flux = ssr_flux + str_flux + sensible_flux + latent_flux
Converting temperatures to Celsius...
t2m: K -> °C
sst: K -> °C
Selecting Variables
Saving: processed_data/daily_era5_chunks/daily_era5_2025_12.nc
Done.

 DONE!
 Processed: 396/396
 Time: 1.2 hours


(396, [])

In [35]:

def combine_era5_chunks():
    """
    Combine all ERA5 monthly chunks (daily processed) into a single 1993-2025 dataset.
    """
    print("\nCombining ERA5 monthly chunks...")
    print("="*65)
    
    chunk_dir = Path("processed_data/daily_era5_chunks")
    files = sorted(chunk_dir.glob("daily_era5_*.nc"))
    print(f"Found {len(files)} files")
    
    if len(files) == 0:
        print("No files found!")
        return None
    
    expected = (2025 - 1993 + 1) * 12  # 372 months
    if len(files) < expected:
        print(f"Warning: Expected {expected} files, found {len(files)}")
        print(f"Missing {expected - len(files)} months")
    
    print(f"First: {files[0].name} ... Last: {files[-1].name}")
    
    
    # ========================================
    # Open and combine
    # ========================================
    print("\nOpening files with lazy loading...")
    try:
        combined = xr.open_mfdataset(
            files,
            combine='nested',           
            concat_dim='time',          
            parallel=False,
            engine='netcdf4',
            chunks={'time': 365}        # Lazy loading with yearly chunks
        )
        print(f"Opened {len(files)} files")
        
    except Exception as e:
        print(f"Error opening files: {e}")        
    
    # ========================================
    # Sort by time
    # ========================================
    print("Sorting by time...")
    combined = combined.sortby('time')
    
    # ========================================
    # Save final dataset
    # ========================================
    output_dir = Path("final_datasets")
    output_dir.mkdir(exist_ok=True)
    output_file = output_dir / "era5_DAILY_1993_2025.nc"
    
    print(f"\nSaving final dataset to {output_file}...")

    combined.to_netcdf(
        output_file, 
        mode='w', 
        format='NETCDF4',
        engine='netcdf4',
    )
    
    combined.close()
    print(f"Saved successfully")
    
    # ========================================
    # Validation
    # ========================================
    print("\nValidating final dataset...")
    
    ds_final = xr.open_dataset(output_file)
    size_mb = output_file.stat().st_size / (1024**2)
    
    start_date = datetime(1993, 1, 1)
    end_date = datetime(2025, 12, 31)
    expected_days = (end_date - start_date).days + 1
    
    print(f"\n{'='*65}")
    print(f"ERA5 DATASET READY!")
    print(f"{'='*65}")
    print(f"File: {output_file}")
    print(f"Size: {size_mb:.0f} MB")
    print(f"Resolution: {ds_final.longitude.values[1] - ds_final.longitude.values[0]:.3f}° (~31 km)")
    print(f"Domain: {ds_final.longitude.min().values:.1f}°E to {ds_final.longitude.max().values:.1f}°E")
    print(f"           {ds_final.latitude.min().values:.1f}°N to {ds_final.latitude.max().values:.1f}°N")
    print(f"Grid: {len(ds_final.longitude)} × {len(ds_final.latitude)} = {len(ds_final.longitude)*len(ds_final.latitude)} points")
    
    print(f"\nTime coverage:")
    print(f"Start: {ds_final.time.values[0]}")
    print(f"End:   {ds_final.time.values[-1]}")
    print(f"Total: {len(ds_final.time)} days")
    print(f"Expected: {expected_days} days")
    
    if len(ds_final.time) == expected_days:
        print(f"Complete dataset!")
    else:
        missing = expected_days - len(ds_final.time)
        print(f"Missing {missing} days")
    
    print(f"\nVariables ({len(ds_final.data_vars)}):")
    for var in ds_final.data_vars:
        print(f"   - {var}")
    
    print(f"{'='*65}")
    
    ds_final.close()
    
    return output_file



In [36]:
final_ear5_file = combine_era5_chunks()


Combining ERA5 monthly chunks...
Found 396 files
First: daily_era5_1993_01.nc ... Last: daily_era5_2025_12.nc

Opening files with lazy loading...
Opened 396 files
Sorting by time...

Saving final dataset to final_datasets/era5_DAILY_1993_2025.nc...
Saved successfully

Validating final dataset...

ERA5 DATASET READY!
File: final_datasets/era5_DAILY_1993_2025.nc
Size: 620 MB
Resolution: 0.250° (~31 km)
Domain: 24.0°E to 36.0°E
           31.0°N to 37.0°N
Grid: 49 × 25 = 1225 points

Time coverage:
Start: 1993-01-01T00:00:00.000000000
End:   2025-12-31T00:00:00.000000000
Total: 12053 days
Expected: 12053 days
Complete dataset!

Variables (11):
   - u10
   - v10
   - t2m
   - msl
   - sst
   - wind_speed
   - ssr_flux
   - str_flux
   - sensible_flux
   - latent_flux
   - net_heat_flux


In [49]:
ds123 = xr.open_dataset("processed_data/daily_era5_chunks/daily_era5_1993_01.nc")
print(ds123)

<xarray.Dataset> Size: 990kB
Dimensions:        (time: 31, latitude: 25, longitude: 29)
Coordinates:
  * time           (time) datetime64[ns] 248B 1993-01-01 ... 1993-01-31
  * latitude       (latitude) float64 200B 36.0 35.75 35.5 ... 30.5 30.25 30.0
  * longitude      (longitude) float64 232B 29.0 29.25 29.5 ... 35.5 35.75 36.0
Data variables:
    u10            (time, latitude, longitude) float32 90kB ...
    v10            (time, latitude, longitude) float32 90kB ...
    t2m            (time, latitude, longitude) float32 90kB ...
    msl            (time, latitude, longitude) float32 90kB ...
    sst            (time, latitude, longitude) float32 90kB ...
    wind_speed     (time, latitude, longitude) float32 90kB ...
    ssr_flux       (time, latitude, longitude) float32 90kB ...
    str_flux       (time, latitude, longitude) float32 90kB ...
    sensible_flux  (time, latitude, longitude) float32 90kB ...
    latent_flux    (time, latitude, longitude) float32 90kB ...
    net_heat

In [50]:

thetao = ds123[["u10", "v10", "t2m", "msl", "sst",'wind_speed','ssr_flux', 'str_flux', 'sensible_flux', 'latent_flux', 'net_heat_flux']]

# Convert to DataFrame
df = thetao.to_dataframe().reset_index()

# ✅ Drop rows with NaN (land points)
# df = df.dropna(subset=['temp_100_150m'])

# Show first few rows (should have NO NaN now)
print(df.head(1000))

          time  latitude  longitude       u10       v10        t2m  \
0   1993-01-01     36.00      29.00 -4.626640  6.231125  12.915375   
1   1993-01-01     36.00      29.25 -4.794121  4.571457  12.638031   
2   1993-01-01     36.00      29.50 -4.593925  3.739182  12.711761   
3   1993-01-01     36.00      29.75 -3.149590  2.369308  12.503754   
4   1993-01-01     36.00      30.00 -1.668877  1.894699  12.881683   
..         ...       ...        ...       ...       ...        ...   
995 1993-01-02     33.75      31.25  3.711170  1.998940  14.557159   
996 1993-01-02     33.75      31.50  2.915760  1.989662  14.757355   
997 1993-01-02     33.75      31.75  1.989979  2.077797  14.742218   
998 1993-01-02     33.75      32.00  0.952869  1.954994  14.881378   
999 1993-01-02     33.75      32.25  0.391102  1.565102  14.890167   

             msl        sst  wind_speed   ssr_flux   str_flux  sensible_flux  \
0    101658.1250  16.527985    7.760974  34.148438 -62.051952     -45.411865   